# Generate Training Images

This script generates dotted/groundtrust image pairs to train, validate and test the model.

In [2]:
# Necessary modules
import random
from pathlib import Path
from typing import Tuple

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from skimage.morphology import skeletonize
import string

import random
from wordfreq import top_n_list, zipf_frequency

## Configuration

In [3]:
# Image width and height
IMG_W, IMG_H = 256, 256

# Font setup
font_size = 80
font_path = r"C:\Windows\Fonts\times.ttf"
FONT = ImageFont.truetype(font_path, font_size)

# Data
#words = list(string.ascii_uppercase + string.ascii_lowercase)  # All upper and lowercase letters
n = 250                            # Number of words to generate per specified length 
word_sizes = [3, 4, 5, 6]          # Word lengths
OUT_DIR = Path("./data")           # output root folder

# Specify train, val, test ratio's
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Dot/"bullet hole" style
DOT_COLOR = 0                      # black in grayscale
RADIUS_RANGE = (1, 3)              # random dot radius
SPACING_RANGE = (3, 8)             # random spacing along skeleton points

# Skeletonization / threshold
THRESH = 200                       # higher => treat more pixels as background

# Optional simple augmentation (kept minimal)
ROTATE_DEG_RANGE = (-8, 8)
SHIFT_PX_RANGE = (-8, 8)

## Functions

In [4]:
def random_common_words(
    seed,          # Seed
    n,             # How many words to create
    l,             # Word length
    total_n=10000, # How many words to sample from
):
    """
    Generates k common english words of length l. A higher n will result in less common words.
    """
    # Set seed
    random.seed(seed)
    
    # Get a list of common words
    words = top_n_list("en", total_n)  # Always generates same words regardless of seed

    # Filter by length and keep alphabetic only (avoid hyphens, apostrophes, etc.)
    words = [w for w in words if w.isalpha() and len(w) == l]
    if not words:
        raise ValueError("No words match length constraints. Try widening min_len/max_len or increasing top_n.")

    return random.sample(words, n)

def crop_and_pack_to_canvas(
    gray: np.ndarray,
    *,
    ink_thresh: int = 245,        # pixels < ink_thresh are considered "ink"
    pad: int = 12,                # padding around bounding box in pixels
    out_size: Tuple[int, int] = (IMG_W, IMG_H),
    fill: int = 255,              # background
    margin: int = 8,              # keep a small border after scaling
) -> np.ndarray:
    """
    1) Find tight bbox around ink pixels
    2) Crop with padding
    3) Resize (keep aspect ratio) so it fills most of out_size
    4) Paste centered on white canvas (no distortion)

    Returns:
        packed grayscale image of shape (out_h, out_w), dtype uint8
    """
    if gray.ndim != 2:
        raise ValueError(f"Expected 2D grayscale image, got shape {gray.shape}")

    out_w, out_h = out_size

    # Identify ink pixels
    ink = gray < ink_thresh
    ys, xs = np.where(ink)

    # If no ink found, return original (or raise)
    if len(xs) == 0 or len(ys) == 0:
        return gray

    # Bounding box
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    # Add padding and clamp
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(gray.shape[1] - 1, x1 + pad)
    y1 = min(gray.shape[0] - 1, y1 + pad)

    crop = gray[y0:y1 + 1, x0:x1 + 1]

    # Resize crop to fit inside (out_w-margin*2, out_h-margin*2) with aspect ratio
    max_w = out_w - 2 * margin
    max_h = out_h - 2 * margin
    ch, cw = crop.shape[:2]

    scale = min(max_w / cw, max_h / ch)
    new_w = max(1, int(round(cw * scale)))
    new_h = max(1, int(round(ch * scale)))

    crop_resized = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Paste centered onto canvas
    canvas = np.full((out_h, out_w), fill, dtype=np.uint8)
    x_start = (out_w - new_w) // 2
    y_start = (out_h - new_h) // 2
    canvas[y_start:y_start + new_h, x_start:x_start + new_w] = crop_resized

    return canvas
    

def text_to_clean_gray(text: str, width: int = IMG_W, height: int = IMG_H) -> np.ndarray:
    """
    Render clean grayscale target
    """
    img = Image.new("L", (width, height), color=255)  # "L" = grayscale
    draw = ImageDraw.Draw(img)

    bbox = draw.textbbox((0, 0), text, font=FONT)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    x = (width - text_w) / 2
    y = (height - text_h) / 2

    draw.text((x, y), text, fill=0, font=FONT)   # black text
    return np.array(img, dtype=np.uint8)         # (H,W), uint8


def augment_affine(gray: np.ndarray) -> np.ndarray:
    """
    Simple augmentation: rotate + shift (grayscale)
    """
    # Ensure grayscale
    if gray.ndim != 2:
        raise ValueError(f"augment_affine expects 2D grayscale array, got shape {gray.shape}")

    h, w = gray.shape
    angle = random.uniform(*ROTATE_DEG_RANGE)
    tx = random.randint(*SHIFT_PX_RANGE)
    ty = random.randint(*SHIFT_PX_RANGE)

    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[0, 2] += tx
    M[1, 2] += ty

    out = cv2.warpAffine(
        gray,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,  # white background
    )
    return out.astype(np.uint8)


def binarise(gray: np.ndarray, thresh: int = THRESH) -> np.ndarray:
    # Invert so text becomes foreground (255)
    _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)
    return bw.astype(np.uint8)


def get_skeleton(binary_img: np.ndarray) -> np.ndarray:
    skel_bool = skeletonize(binary_img > 0)  # boolean foreground
    return (skel_bool.astype(np.uint8) * 255)


def dotted_from_skeleton(
    skeleton: np.ndarray,
    canvas_shape: Tuple[int, int],
    radius: int,
    spacing: int,
    dot_color: int = DOT_COLOR,
) -> np.ndarray:
    """
    Draw bullet-hole dots along skeleton (grayscale)
    """
    canvas = np.full(canvas_shape, 255, dtype=np.uint8)  # white background

    pts = np.column_stack(np.where(skeleton > 0))  # (y, x)
    if pts.size == 0:
        return canvas

    # Simple deterministic ordering
    pts = pts[np.lexsort((pts[:, 1], pts[:, 0]))]

    for i in range(0, len(pts), spacing):
        y, x = pts[i]
        cv2.circle(canvas, (int(x), int(y)), int(radius), int(dot_color), thickness=-1)

    return canvas


def thicken_edges(edge: np.ndarray, k: int = 1) -> np.ndarray:
    """
    Dilate a binary edge map by k iterations to make edges thicker.
    edge: uint8 {0,255}
    """
    if k <= 0:
        return edge
    kernel = np.ones((3, 3), np.uint8)
    return cv2.dilate(edge, kernel, iterations=k)
    

def save_grayscale(path: Path, gray: np.ndarray):
    """
    Save helper that guarantees saving a single-channel grayscale image.
    """
    if gray.ndim != 2:
        raise ValueError(f"save_grayscale expects 2D grayscale array, got shape {gray.shape}")

    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(gray.astype(np.uint8), mode="L").save(str(path))

In [5]:
class ImageGenerator:
    def __init__(self, start_idx=0):
        self.idx = start_idx

    def generate_images(self, words, v, dir_name):
        """
        Generates and saves word images
        """
        count = 0
        for word in words:
            for _ in range(v):
                # 1) clean target (grayscale)
                target = text_to_clean_gray(word)
                target = augment_affine(target)
                
                # 1b) tight crop + scale up + repack to 256x256
                target = crop_and_pack_to_canvas(
                    target,
                    ink_thresh=245,
                    pad=12,
                    out_size=(IMG_W, IMG_H),
                    margin=8
                )
                
                # 2) skeletonize from the packed target
                bw = binarise(target)
                skel = get_skeleton(bw)
                skel_save = thicken_edges(skel, k=1)  # Thicken skeleton edges
               
                # 3) dotted input (grayscale) – same shape as target (256x256)
                radius = random.randint(*RADIUS_RANGE)
                spacing = random.randint(*SPACING_RANGE)
                inp = dotted_from_skeleton(skel, target.shape, radius=radius, spacing=spacing)
        
                # 4) save pair with matching name
                input_dir = OUT_DIR / f"{dir_name}_input" 
                input_path = input_dir / f"{word}_{self.idx:06d}_input.png"
                skeleton_dir = OUT_DIR / f"{dir_name}_skeleton"
                skeleton_path = skeleton_dir / f"{word}_{self.idx:06d}_skeleton.png"
                target_dir = OUT_DIR / f"{dir_name}_target" 
                target_path = target_dir / f"{word}_{self.idx:06d}_target.png"
        
                save_grayscale(input_path, inp)
                save_grayscale(skeleton_path, skel_save)
                save_grayscale(target_path, target)
        
                self.idx += 1
                count += 1

        print(f"✅ Done. Wrote {count} image pairs to: {input_dir.resolve()} and {target_dir.resolve()}")

## Main

In [6]:
def main(seed):
    """
    Main function that handles everything
    """
    # Generate words
    words = []
    for size in word_sizes:
        words.append(random_common_words(seed=seed, n=n, l=size))

    # Subset words into lists for train, val and test, following the specified ratio's
    train_n = round(n*train_ratio)
    val_n = round(n*val_ratio)
    test_n = round(n*test_ratio)  # Watch out, this can break the script if n is less than 10
    
    words_train = [word for sublist in words for word in sublist[0:train_n]]
    words_val = [word for sublist in words for word in sublist[train_n:train_n+val_n]]
    words_test = [word for sublist in words for word in sublist[-test_n:]]

    # Initialize generator class
    generator = ImageGenerator()

    # Generate training image pairs
    generator.generate_images(words=words_train, v=1, dir_name='train_1')
    generator.generate_images(words=words_train, v=3, dir_name='train_3')
    generator.generate_images(words=words_train, v=5, dir_name='train_5')

    # Generate validation image pairs
    generator.generate_images(words=words_val, v=2, dir_name='val')

    # Generate testing image pairs
    generator.generate_images(words=words_test, v=2, dir_name='test')

In [7]:
if __name__ == "__main__":
    main(seed=5)

✅ Done. Wrote 800 image pairs to: C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_1_input and C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_1_target
✅ Done. Wrote 2400 image pairs to: C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_3_input and C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_3_target
✅ Done. Wrote 4000 image pairs to: C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_5_input and C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\train_5_target
✅ Done. Wrote 200 image pairs to: C:\Users\bzambeek001\OneDrive - PwC\Documents\Learning\AI\Deep Neural Engineering\Assignment 2\data\val_input and C:\Users\bzambeek001\OneDrive - PwC\Documents\Lea